<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading_V0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V0-B Initialized Fixed Grid Backtest

V0-B is a fixed arithmetic **spot grid** prepared for real-trading workflow design.

At startup, the strategy:
- keeps USDT for grid BUY levels that are still below the starting market,
- buys only the BTC quantity required to seed SELL targets above the starting market,
- sizes every seeded BTC position so its BTC quantity matches the quantity that the same grid would receive from a normal future BUY.

This avoids over-sizing the initial BTC inventory.

**Execution assumptions**
- Initial BTC inventory is created at the first candle Open.
- Existing SELL targets are processed before new BUYs in each 1-minute candle.
- BUY occurs only when price crosses a free grid level downward.
- A new downward-crossing BUY cannot SELL in the same candle.
- A level sold in the current candle cannot rebuy in that candle.
- Same-candle SELL proceeds are not reused for BUYs.

This notebook is still a **backtest / shadow-readiness model**. It does not send Binance orders.


## 0. Setup

Imports and Google Drive connection only. Trading logic is kept together in Section 1.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import base64
import bisect
import heapq
import json
import os

import numpy as np
import pandas as pd
import requests


# 1. Trading System

Everything that defines **how the strategy trades** is grouped in this section:
configuration, grid construction, initialization sizing, position economics, and the trading engine.


## 1.1 Trading Configuration

These are the parameters a trader changes.


In [ ]:
# Trading instrument
SYMBOL = "BTCUSDT"

# Trading capital
INITIAL_CAPITAL = 3000.0

# Fixed arithmetic grid
GRID_FLOOR = 38000.0
GRID_CEILING = 127000.0
GRID_GAP = 1000.0

# Trading fees
BUY_FEE = 0.001
SELL_FEE = 0.001


## 1.2 Grid Geometry

A grid slot is:

`BUY at buy_price  →  SELL at sell_target`

The order size is derived later because V0-B initialization depends on the actual starting market price.


In [ ]:
def validate_trading_config(capital, floor, ceiling, gap, buy_fee, sell_fee):
    if capital <= 0: raise ValueError("INITIAL_CAPITAL must be greater than 0.")
    if floor <= 0: raise ValueError("GRID_FLOOR must be greater than 0.")
    if ceiling <= floor: raise ValueError("GRID_CEILING must be greater than GRID_FLOOR.")
    if gap <= 0: raise ValueError("GRID_GAP must be greater than 0.")
    if not (0 <= buy_fee < 1): raise ValueError("BUY_FEE must be in [0, 1).")
    if not (0 <= sell_fee < 1): raise ValueError("SELL_FEE must be in [0, 1).")
    raw_grid_count = (ceiling - floor) / gap
    if not np.isclose(raw_grid_count, round(raw_grid_count)):
        raise ValueError("(GRID_CEILING - GRID_FLOOR) must be exactly divisible by GRID_GAP.")
    return int(round(raw_grid_count))

def build_grid_template(floor, ceiling, gap):
    buy_prices = np.arange(floor, ceiling, gap, dtype=float)
    return pd.DataFrame({"grid_id": np.arange(1, len(buy_prices) + 1), "buy_price": buy_prices, "sell_target": buy_prices + gap})

NUMBER_OF_GRIDS = validate_trading_config(INITIAL_CAPITAL, GRID_FLOOR, GRID_CEILING, GRID_GAP, BUY_FEE, SELL_FEE)
GRID_TEMPLATE = build_grid_template(GRID_FLOOR, GRID_CEILING, GRID_GAP)
print(f"Number of Grids : {NUMBER_OF_GRIDS}")


## 1.3 Initialized Portfolio Sizing


In [ ]:
def derive_initialized_grid(grid_template, start_price, initial_capital, buy_fee):
    grid = grid_template.copy()
    grid_floor = float(grid["buy_price"].min())
    grid_ceiling = float(grid["sell_target"].max())
    if not (grid_floor < start_price < grid_ceiling):
        raise ValueError("Starting market price must be strictly inside the configured grid range.")
    seed_mask = grid["sell_target"] > start_price
    reserve_mask = ~seed_mask
    seed_buy_prices = grid.loc[seed_mask, "buy_price"].to_numpy(float)
    funding_weight = int(reserve_mask.sum()) + float(np.sum(start_price / seed_buy_prices))
    if funding_weight <= 0: raise ValueError("Invalid initialization funding weight.")
    normal_order_size = initial_capital / funding_weight
    grid["order_size_usdt"] = float(normal_order_size)
    grid["seed_at_start"] = seed_mask
    grid["normal_net_btc"] = normal_order_size / grid["buy_price"] * (1.0 - buy_fee)
    grid["initial_gross_btc"] = np.where(seed_mask, grid["normal_net_btc"] / (1.0 - buy_fee), 0.0)
    grid["initial_entry_cost_usdt"] = np.where(seed_mask, grid["initial_gross_btc"] * start_price, 0.0)
    reserved_cash = float(reserve_mask.sum() * normal_order_size)
    seeded_btc_cost = float(grid["initial_entry_cost_usdt"].sum())
    if not np.isclose(reserved_cash + seeded_btc_cost, initial_capital, atol=1e-8):
        raise AssertionError("Initialization sizing does not reconcile to INITIAL_CAPITAL.")
    sizing = {"normal_order_size_usdt": float(normal_order_size), "funding_weight": float(funding_weight), "initial_sell_positions": int(seed_mask.sum()), "initial_buy_levels": int(reserve_mask.sum()), "reserved_cash_usdt": reserved_cash, "seeded_btc_cost_usdt": seeded_btc_cost}
    return grid, sizing


# 2. Backtest System

The trading engine remains unchanged from the validated V0-B baseline.


## Notice

The previous compressed benchmark revision has been rolled back because readability and reproducibility take priority. Re-run the stable V0-B baseline first; the Buy & Hold benchmark will be reintroduced as an isolated backtest-only cell without changing the Trading System.
